# Getting Started with Jupyter
The first step of using a Jupyter notebook is selecting your environment.

Selecting an environment means you can make use of all of the software in it for your Jupyter notebook, and effectively tells your computer what it has to work with when you ask it for a given function.

To get started, click the environment button in the top right, and select the 2026_figure environment we made in Lesson 0.

Anytime you want to run some code, select the block you want to run, and hit the run button to its left. Hitting `Ctrl + Enter` on your keyboard also runs the block you've got selected. Usually in a Jupyter notebook, blocks depend on the ones above them, so they may not run, or may not run the way you'd expect if the previous ones haven't been run.

## The Import Block
In Python, the first thing you'll usually see in any script or Jupyter notebook is a bunch of lines saying `import xyz_package` and `from xyz_package import a_useful_function`.

The purpose of these is to take software from your environment and let the file you're working on make use of it. This usually includes a blend of packages from Python's standard library -- all of the useful things that ship with Python itself -- alongside additional software tools like Polars, SciPy, and Matplotlib.

Go ahead and run the import block below.

In [ ]:
# This line is a comment. It lets you add commentary directly next to your code.
# That's always useful to people looking at it later on, so you should use these liberally to explain your thought processes on why you did something.
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy.stats import f_oneway as anova
from scipy.stats import false_discovery_control
from scipy.stats import kruskal as kruskal_wallis
from scipy.stats import shapiro as shapiro_wilk

# You'll notice that in this import block, I frequently import something as something else.
# That's a trick you can use to make shorthand names for software or to clarify exactly what something is or where it comes from.

---

# Introduction to Polars
As I mentioned, Polars effectively lets you do most things that you could do with Excel in code form. It broadly exists to load, manipulate, and view data.

For today, we'll be working with the [Iris Dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set), one of the most famous datasets in statistics and machine learning. It's a relatively small set of observations about three different types of irises, which makes it perfect as a tool for learning the basics.

In [ ]:
# This is what's called a Constant. Basically it's a convenient label that we set once and never change.
# Typically, these are written in SCREAMING_SNAKE_CASE (Yes that's the name) to differentiate them from normal variables.
# This basically lets us refer to wherever our data is as just "DATA_PATH" in later code, without needing to type out its exact location every time.
# Most data analysis starts with a block like this that declares where your data is, and any other useful constants, to make referencing them easy later.
DATA_PATH = Path("iris_dataset.tsv")

In [ ]:
# This reads our data from the file we just specified, using Tabs to determine where the columns are
data = pl.read_csv(DATA_PATH, separator="\t")

# The "display" function takes some input, and prints it nicely
display(data)

## Exercise 1
The first thing we want to do is to get a sense of what's present in our dataset. To that end, the block below does the following:
- Displays the first 5 entries in the dataset.
- Displays the unique types of irises present and how many of each we have.
- Displays the minimum, maximum, and mean for each of our numeric columns.

Once you run the block and observe the results, try to add the following things:
- Display the **last** 10 entries in the dataset.
- Display the **median** for each of our numeric columns.

If you get stuck, try looking at the steps in the **A Brief Guide on Debugging** section of today's lesson, particularly steps 2 and 3.

In [ ]:
display(data.head(n=5))
display(data.select(pl.col("iris type").value_counts()).unnest("iris type"))

display(data.select(cs.exclude(pl.col("iris type"))).min())
display(data.select(cs.exclude(pl.col("iris type"))).max())
display(data.select(cs.exclude(pl.col("iris type"))).mean())

# This is where you'll add some lines. As a hint, you can get pretty far here by copying and pasting similar lines from up above.

---

# Introduction to SciPy
Now that we've taken a look at some summary statistics about our dataset, let's dive into it a bit more and try and learn something about it.

We'll start with the `sepal length (cm)` column. We want to test the hypothesis that sepal length is a good way to tell different types of irises apart.

As this is an important research question, we want to ensure that there's only a 5% chance that the Null Hypothesis is true, so we want a p-value of 0.05 or less.

In [ ]:
# This is the threshold we'll use to determine if a result is significant or not
ALPHA = 0.05

## Exercise 2

What is our Null Hypothesis? (You can type your answer by adding it to this block and re-running it.)

## Exercise 3

Now that we have our Null Hypothesis and our desired significance level, let's determine what kind of test we'll use by following these steps.
1. First, we want to see if we can use parametric tests on our data. You can run the block below, and it will say if the data is normally distributed or not.
2. Afterwards, fill in the appropriate test to use in the following code block. (As we have 3 groups, it should be either an ANOVA or Kruskal-Wallis test.)
3. Finally, run the test you chose, and determine if we should reject the Null Hypothesis.

In [ ]:
normality_p_values = []

# This section of the code is iterating through all of the types of irises in our dataset
for iris_type in data.select("iris type").unique().to_series().to_list():
    # It then calculates the likelihood that the data for that iris type is normally distributed
    __, p_value = shapiro_wilk(data.filter(pl.col("iris type").eq(iris_type)).select("sepal length (cm)"))
    normality_p_values.append(p_value)

# Finally, we apply a multiple hypothesis correction, and check if  our datasets are normally distributed.
adjusted_normality_p_values = false_discovery_control(normality_p_values, method="bh")

# BONUS QUESTION: Why is it that we're checking if all of the p values are above our alpha threshold here? What is our Null Hypothesis for the Shapiro Wilk test?
if all(p_val >= ALPHA for p_val in adjusted_normality_p_values):
    print("All datasets are normally distributed.")
else:
    print("Not all datasets are normally distributed.")

In [ ]:
# Here you'll want to replace "your_statistical_test" with the test you chose
# HINT: You can see the names of the tests we imported back in the import block
__, p_value = your_statistical_test(
    data.filter(pl.col("iris type").eq("Setosa")).select("sepal length (cm)"),
    data.filter(pl.col("iris type").eq("Virginica")).select("sepal length (cm)"),
    data.filter(pl.col("iris type").eq("Versicolour")).select("sepal length (cm)"),
)
if p_value <= ALPHA:
    print("We should reject the Null Hypothesis.")
else:
    print("We should not reject the Null Hypothesis.")

---

# Introduction to Matplotlib
Now that we've seen if there's a significant difference or not in sepal length, let's actuallly visualize what the distributions look like using Seaborn and Matplotlib. (Seaborn largely is just a set of convenience functions which make matplotlib easier to use.)

Run the cell below, and then decide if it looks like there's a difference to you or not.

In [ ]:
# This line sets the default font size. You can customize many other defaults similarly.
plt.rcParams["font.size"] = 14

# This line creates a blank figure
fig, ax = plt.subplots(figsize=(6, 8))

# These next lines actually display the data, giving us both a box and whiskers plot showing the general data trends, as well as a swarmplot showing individual data points
sns.boxplot(data=data, x="iris type", y="sepal length (cm)", fill=False, color="black")
sns.swarmplot(data=data, x="iris type", y="sepal length (cm)", color="blue", edgecolor="black", linewidth=1, size=7)

# These lines add nicer looking x and y labels, as well as a title to the figure
plt.xlabel("Species", fontdict={"fontsize": 16})
plt.ylabel("Sepal Length (cm)", fontdict={"fontsize": 16})
plt.title("Sepal Length by Iris Species", fontdict={"fontsize": 18})

# This cleans up leftover space, and then displays the figure
plt.tight_layout()
plt.show()

---

# Challenges
Now that you've tried it with one feature of our iris dataset, try going through the full process with two more features besides sepal length.

See if you can do the following in the code block below:
1. Plot the distributions of the three iris types for the features you choose.
2. Determine if you should use an ANOVA or the Kruskal-Wallis test to determine if the features can tell the iris types apart.
3. Determine if you should reject the Null Hypothesis or not for each feature at an Alpha of 0.05. What about at an Alpha of 0.01?

**BONUS CHALLENGES**
1. Plot a violin plot instead of the boxplot and swarm plot.
2. Plot a 2d scatter plot of the two features you chose, and color the dots based on the iris type.